In [ ]:
# ============================================================
# 06_robustness.ipynb
# Model-risk / robustness analysis: inter-model consistency of
# DiCE counterfactual recommendations across the top tree-based
# models (XGBoost, LightGBM, CatBoost).
#
# For each high-risk individual, DiCE is run on each model and the
# sets of changed features are compared pairwise with the Jaccard
# coefficient. Low agreement implies recommendations should be
# aggregated across models before being surfaced to a policyholder.
#
# Input : data/processed/models_{sex}/{xgb,lgb,cat}_pipe.pkl,
#         data/processed/scored_{sex}.csv,
#         data/processed/highrisk_nondiabetic_{sex}.csv
# Output: results/tables/robustness_consistency.csv,
#         results/figures/robustness_heatmap.(png|pdf)
#
# Set env N_CASES to cap individuals per sex (default 10, as in paper).
# Figures grayscale, dpi 600, PNG + PDF, no captions.
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import dice_ml

SEED = 42
np.random.seed(SEED)
N_CASES = int(os.environ.get("N_CASES", "10"))
TOTAL_CFS = 4

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="white", context="paper")
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 10, "image.cmap": "Greys",
})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

cat_col = ["HE_obe","BO1_1","BO1_2","BO1_3","BD1_11","BD2_1","BS3_1","BE3_71",
           "BE3_75","BE3_81","BE3_91","pa_aerobic","L_BR_FQ","BP1","mh_stress",
           "incm","ho_incm","edu","BH1"]
num_col = ["HE_BMI","HE_wc","HE_wt","N_SUGAR","N_CHO","N_EN","age"]
feature_col = cat_col + num_col
TARGET = "HE_DM_HbA1c"
IMMUTABLE = ["age"]
vary = [c for c in feature_col if c not in IMMUTABLE]
MODELS = ["xgb", "lgb", "cat"]

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 1.0

def changed_set(row, cfX, j):
    ch = []
    for c in feature_col:
        if c in IMMUTABLE:
            continue
        if float(cfX.iloc[j][c]) != float(row[c].iloc[0]):
            ch.append(c)
    return ch

rows, heat = [], {}
for sex in ["male", "female"]:
    train = pd.read_csv(PROC / f"scored_{sex}.csv")[feature_col + [TARGET]]
    hr = pd.read_csv(PROC / f"highrisk_nondiabetic_{sex}.csv")[feature_col].reset_index(drop=True).iloc[:N_CASES]
    dd = dice_ml.Data(dataframe=train, continuous_features=num_col, outcome_name=TARGET)
    pr = {c: [float(train[c].min()), float(train[c].max())] for c in num_col if c not in IMMUTABLE}

    exps = {}
    for mk in MODELS:
        pipe = joblib.load(PROC / f"models_{sex}" / f"{mk}_pipe.pkl")
        exps[mk] = dice_ml.Dice(dd, dice_ml.Model(model=pipe, backend="sklearn"), method="genetic")

    pair_names = ["xgb-lgb", "xgb-cat", "lgb-cat"]
    mat = np.full((len(hr), len(pair_names)), np.nan)
    for i in range(len(hr)):
        row = hr.iloc[[i]]
        chsets = {}
        for mk in MODELS:
            try:
                cf = exps[mk].generate_counterfactuals(
                    row, total_CFs=TOTAL_CFS, desired_class=0,
                    features_to_vary=vary, permitted_range=pr, verbose=False)
                cfX = cf.cf_examples_list[0].final_cfs_df[feature_col]
                # use the first CF's changed set for consistency comparison
                chsets[mk] = changed_set(row, cfX, 0)
            except Exception:
                chsets[mk] = []
        for pj, (a, b) in enumerate([("xgb", "lgb"), ("xgb", "cat"), ("lgb", "cat")]):
            if chsets[a] or chsets[b]:
                mat[i, pj] = jaccard(chsets[a], chsets[b])
    heat[sex] = mat
    case_mean = np.nanmean(mat, axis=1)
    rows.append(dict(Sex=sex, Cases=len(hr),
                     Mean_consistency=round(np.nanmean(mat), 3),
                     SD=round(np.nanstd(mat), 3),
                     Min=round(np.nanmin(case_mean), 3),
                     Max=round(np.nanmax(case_mean), 3)))
    print(f"{sex:7s}: mean Jaccard consistency = {np.nanmean(mat):.3f} (n={len(hr)} cases)")

cons = pd.DataFrame(rows)
cons.to_csv(TAB / "robustness_consistency.csv", index=False)
print("\nRobustness consistency:\n", cons.to_string(index=False))

# ---- Figure: heatmap of pairwise consistency (cases x model-pairs) ----
pair_names = ["XGB-LGB", "XGB-Cat", "LGB-Cat"]
fig, axes = plt.subplots(1, 2, figsize=(8.5, 5.2))
for ax, sex in zip(axes, ["male", "female"]):
    m = heat[sex]
    hm = sns.heatmap(m, ax=ax, cmap="Greys", vmin=0, vmax=1, cbar=True,
                     linewidths=0.4, linecolor="0.8",
                     xticklabels=pair_names,
                     yticklabels=[f"C{i+1}" for i in range(m.shape[0])])
    ax.set_xlabel(f"Model pair — {sex.capitalize()}")
    ax.set_ylabel("High-risk case")
save_fig(fig, "robustness_heatmap")
print("Figure written: robustness_heatmap (png+pdf).")
